In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)


WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Arjun Prithvi\\LangChain\\myenv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200))

In [ ]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

loader = WebBaseLoader("https://reference.langchain.com/python/langchain")
docs = loader.load()

chunking = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = chunking.split_documents(docs)

vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever()


c:\Users\Arjun Prithvi\LangChain\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1564.74it/s]


In [45]:
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(retriever,"LangChain_Documentation","LangChain Documentation Retriever")

In [46]:
#combining all te tools into a list

tools = [wiki,arxiv,retriever_tool]

In [47]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
    
llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash",api_key=os.getenv("GEMMA_APIKEY"))


In [48]:
from langsmith import Client

hub = Client()
prompt = hub.pull_prompt("hwchase17/openai-functions-agent", dangerously_pull_public_prompt=True)
type(prompt)


langchain_core.prompts.chat.ChatPromptTemplate

In [49]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the available tools when necessary."
)

In [ ]:
response = agent.invoke({
    "messages": [
        ("user", "What is Langchain?")
    ]
})

response

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: lsv2_********************************************d9
Failed to refresh cache entry hwchase17/openai-functions-agent: Connection error caused failure to GET /commits/hwchase17/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/openai-functions-agent/latest (Caused by NameRes